In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import os
import urllib.request

# Ensure the bronze directory exists
bronze_dir = "/lakehouse/default/Files/bronze"
os.makedirs(bronze_dir, exist_ok=True)

# Download NYC Taxi January 2024 trip data
taxi_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
taxi_path = f"{bronze_dir}/yellow_tripdata_2024-01.parquet"

urllib.request.urlretrieve(taxi_url, taxi_path)
file_size_mb = round(os.path.getsize(taxi_path) / (1024 * 1024), 2)
print(f"NYC Taxi file downloaded successfully! Size: {file_size_mb} MB")

StatementMeta(, dcae8e83-5ed3-454d-af82-340dff511cef, 3, Finished, Available, Finished, False)

NYC Taxi file downloaded successfully! Size: 47.65 MB


In [2]:
%%sql
CREATE OR REPLACE TABLE silver_nyc_taxi AS
SELECT
    *,
    CAST(tpep_pickup_datetime AS DATE) AS pickup_date
FROM parquet.`Files/bronze/yellow_tripdata_2024-01.parquet`
WHERE CAST(tpep_pickup_datetime AS DATE) BETWEEN '2024-01-01' AND '2024-01-31'
  AND trip_distance > 0
  AND total_amount > 0
  AND passenger_count > 0;

StatementMeta(, dcae8e83-5ed3-454d-af82-340dff511cef, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
%%sql
CREATE OR REPLACE TABLE gold_taxi_daily_summary AS
SELECT
    pickup_date,
    CAST(COUNT(*) AS INT) AS total_trips,
    CAST(ROUND(SUM(total_amount), 2) AS DOUBLE) AS total_revenue,
    CAST(ROUND(AVG(trip_distance), 2) AS DOUBLE) AS avg_distance,
    CAST(ROUND(AVG(total_amount), 2) AS DOUBLE) AS avg_fare
FROM silver_nyc_taxi
GROUP BY pickup_date
ORDER BY pickup_date;

StatementMeta(, dcae8e83-5ed3-454d-af82-340dff511cef, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
SELECT * FROM gold_taxi_daily_summary LIMIT 10;

StatementMeta(, dcae8e83-5ed3-454d-af82-340dff511cef, 6, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 5 fields>